# 🏙️ Traffy Fondue Machine Learning: Standalone 3-Tier Pipeline
### ระบบ AI ครบวงจรพยากรณ์ระยะเวลาซ่อมแซมและจัดลำดับความสำคัญปัญหาเมือง กทม.
*(ฉบับ All-in-One: รวมโค้ดครบในไฟล์เดียว อ่านข้อมูลจากโฟลเดอร์ในเครื่อง 100% ไม่ต้องดึง API)*

---

### 🏛️ สถาปัตยกรรมระบบ 3 ชั้น (3-Tier Architecture):
```text
[ข้อมูลจริงในโฟลเดอร์ data/processed/ หรือ data/raw/]
                         │
                         ▼
┌────────────────────────────────────────────────────────┐
│ Tier 1: Local LLM Feature Extraction (Ollama: qwen3:4b) │
│ - สกัด Severity (1-5), Sub-category, และ Predicted Dept│
└────────────────────────┬───────────────────────────────┘
                         │
                         ▼
┌────────────────────────────────────────────────────────┐
│ Tier 2: Specialized Sub-models (Mixture of Experts)   │
│ - Router ส่งงานแยกตามความเชี่ยวชาญของแต่ละฝ่าย (5 ฝ่าย)│
│ - LightGBM Regressor ทำนายระยะเวลาซ่อมเสร็จ (วัน)      │
└────────────────────────┬───────────────────────────────┘
                         │
                         ▼
┌────────────────────────────────────────────────────────┐
│ Tier 3: Actionable Decision Layer (BMA Action Matrix)  │
│ - Public Impact Score = Severity × Density × Days      │
│ - จัดกลุ่ม Triage 4 Quadrants สั่งการทันที (เช่น 24 ชม.) │
└────────────────────────────────────────────────────────┘
```


In [ ]:
import os
import sys
import glob
import json
import time
import re
import urllib.request
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, median_absolute_error

# กำหนดเส้นทางโฟลเดอร์หลักในเครื่อง (Local Directories)
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
SPECIALIZED_DIR = os.path.join(MODELS_DIR, "specialized")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SPECIALIZED_DIR, exist_ok=True)

print("✅ สภาพแวดล้อมพร้อมใช้งาน (รันแบบ Local 100%)")
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {PROCESSED_DIR}")


---
## 1. การโหลดข้อมูลจากโฟลเดอร์ในเครื่อง (Local Data Loading)
อ่านข้อมูลที่คลีนแล้วจากโฟลเดอร์ `data/processed/` ในเครื่องโดยตรง (ไม่ต้องดึง API ภายนอก):
* **Train Set (ปี 2023):** `data/processed/train_cleaned.parquet`
* **Validation Set (ต้นปี 2024):** `data/processed/val_cleaned.parquet`
* **Test Set (กลางปี 2024):** `data/processed/test_cleaned.parquet`


In [ ]:
train_path = os.path.join(PROCESSED_DIR, "train_cleaned.parquet")
val_path = os.path.join(PROCESSED_DIR, "val_cleaned.parquet")
test_path = os.path.join(PROCESSED_DIR, "test_cleaned.parquet")

# 1. ตรวจสอบและโหลดจาก data/processed/ (Parquet / CSV)
if os.path.exists(train_path):
    print("📂 กำลังอ่านชุดข้อมูลจากโฟลเดอร์ data/processed/...")
    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)
    test_df = pd.read_parquet(test_path)
elif os.path.exists(os.path.join(PROCESSED_DIR, "train_cleaned.csv")):
    print("📂 กำลังอ่านชุดข้อมูลจาก data/processed/*.csv...")
    train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_cleaned.csv"), low_memory=False)
    val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "val_cleaned.csv"), low_memory=False)
    test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test_cleaned.csv"), low_memory=False)
else:
    raise FileNotFoundError(f"ไม่พบไฟล์ข้อมูลใน {PROCESSED_DIR} กรุณาตรวจสอบโฟลเดอร์ data/processed/")

print(f"📊 1. Train Set (ปี 2023)         : {len(train_df):,} แถว")
print(f"📊 2. Validation Set (ต้นปี 2024): {len(val_df):,} แถว")
print(f"📊 3. Test Set (กลางปี 2024)     : {len(test_df):,} แถว")
print(f"--> รวมทั้งหมด                    : {len(train_df) + len(val_df) + len(test_df):,} แถว")

# แสดงตัวอย่างข้อมูล 5 แถวแรก
train_df[['ticket_id', 'district', 'main_type', 'sub_category', 'predicted_dept', 'severity', 'duration_days']].head()


---
## 2. ตัวแปรต้น 11 Features หน้างาน (The 11 Front-End Features)
ตัวแปรที่ระบบสามารถทราบได้ **ณ วินาทีแรกที่ประชาชนกดแจ้งเรื่อง** โดยไม่ใช้ข้อมูลในอนาคต (No Data Leakage)


In [ ]:
feature_cols = [
    'district', 'main_type', 'sub_category', 'predicted_dept',
    'severity', 'comment_len', 'day_of_week', 'is_weekend',
    'month', 'is_rainy_season', 'hour'
]
cat_cols = ['district', 'main_type', 'sub_category', 'predicted_dept']

print("11 Features หน้างาน (ตัวแปร X):")
for i, col in enumerate(feature_cols, 1):
    c_type = "Categorical" if col in cat_cols else ("Binary" if "is_" in col else "Numerical")
    print(f"  {i:2d}. {col:<18} | ประเภท: {c_type:<12} | ค่าไม่ซ้ำ: {train_df[col].nunique()}")

# สถิติของ Target (duration_days)
print("
สถิติของ Target (ระยะเวลาซ่อมเสร็จจริง - วัน):")
print(train_df['duration_days'].describe(percentiles=[0.25, 0.50, 0.75, 0.90]))


---
## 3. [TIER 1] Local LLM (Ollama) & Feature Extraction Engine
คลาส `LLMComplaintAnalyzer` สกัด **Severity (1-5)**, **Sub-category** และ **Predicted Department**
*(มีระบบ Fast Heuristic สแกนคำสำคัญภาษาไทย ทำงานทดแทนอัตโนมัติหากไม่ได้เปิด Ollama ทำให้รันได้ลื่นไหล ไม่ค้าง)*


In [ ]:
VALID_DEPARTMENTS = [
    "ฝ่ายโยธา", "ฝ่ายรักษาความสะอาดฯ", "ฝ่ายเทศกิจ",
    "สำนักการระบายน้ำ", "ฝ่ายสิ่งแวดล้อมฯ", "การไฟฟ้านครหลวง",
    "การประปานครหลวง", "สำนักการจราจรและขนส่ง", "สำนักงานเขตทั่วไป"
]

SYSTEM_PROMPT = """คุณคือ AI ผู้เชี่ยวชาญด้านการจำแนกปัญหาเมืองและงานปฏิบัติการของกรุงเทพมหานคร (กทม.)
ให้อ่านข้อความร้องเรียนจากประชาชน แล้วตอบกลับเป็น JSON วิเคราะห์ 4 อย่าง:
1. severity: 1-5 (1=สวยงาม, 2=กวนใจเล็กน้อย, 3=ปานกลาง, 4=อันตราย/จราจรติดขัด, 5=วิกฤต/อันตรายถึงชีวิต)
2. sub_category: เนื้องานย่อย (เช่น 'ฝาท่อชำรุด', 'ถนนเป็นหลุมบ่อ', 'ขยะตกค้าง', 'ไฟฟ้าดับ')
3. predicted_dept: ฝ่ายรับผิดชอบหลัก เลือก 1 จากรายการนี้:
   ['ฝ่ายโยธา', 'ฝ่ายรักษาความสะอาดฯ', 'ฝ่ายเทศกิจ', 'สำนักการระบายน้ำ', 'ฝ่ายสิ่งแวดล้อมฯ', 'การไฟฟ้านครหลวง', 'การประปานครหลวง', 'สำนักการจราจรและขนส่ง', 'สำนักงานเขตทั่วไป']
4. reason: เหตุผลสั้นๆ 1 ประโยค
ตอบเฉพาะ JSON: {"severity": <1-5>, "sub_category": "<งานย่อย>", "predicted_dept": "<ฝ่าย>", "reason": "<เหตุผล>"}"""

class LLMComplaintAnalyzer:
    def __init__(self, ollama_model: str = "qwen3:4b", ollama_host: str = "http://localhost:11434", timeout: float = 2.0):
        self.ollama_model = ollama_model
        self.ollama_url = f"{ollama_host}/api/generate"
        self.timeout = timeout

    def analyze(self, text: str) -> dict:
        if not text or not isinstance(text, str) or len(text.strip()) == 0:
            return {"severity": 2, "sub_category": "ทั่วไป", "predicted_dept": "สำนักงานเขตทั่วไป", "reason": "ไม่มีข้อความ", "engine": "default"}
        
        prompt = f"{SYSTEM_PROMPT}\n\nข้อความร้องเรียน: \"{text.strip()}\"\nตอบกลับเฉพาะ JSON:"
        payload = {"model": self.ollama_model, "prompt": prompt, "stream": False, "format": "json"}
        
        try:
            req = urllib.request.Request(self.ollama_url, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                match = re.search(r'\{.*?\}', data.get("response", ""), re.DOTALL)
                if match:
                    parsed = json.loads(match.group(0))
                    sev = max(1, min(5, int(parsed.get("severity", 3))))
                    sub_cat = str(parsed.get("sub_category", "ทั่วไป")).strip() or "ทั่วไป"
                    dept = str(parsed.get("predicted_dept", "สำนักงานเขตทั่วไป")).strip()
                    dept = dept if dept in VALID_DEPARTMENTS else "สำนักงานเขตทั่วไป"
                    return {"severity": sev, "sub_category": sub_cat, "predicted_dept": dept, "reason": parsed.get("reason", "วิเคราะห์โดย Ollama"), "engine": f"ollama ({self.ollama_model})"}
        except Exception:
            pass
            
        c = text.lower()
        sev = 2
        sub_cat = "ทั่วไป"
        dept = "สำนักงานเขตทั่วไป"
        
        if re.search(r'ฝาท่อ|ท่อระบาย|น้ำท่วม|น้ำขัง|ลอกท่อ|ระบายน้ำ', c):
            sub_cat = "ฝาท่อชำรุด" if "ฝาท่อ" in c else "น้ำท่วมขัง"
            dept = "สำนักการระบายน้ำ" if "ระบายน้ำ" in c or "ลอกท่อ" in c else "ฝ่ายโยธา"
        elif re.search(r'ขยะ|ถังขยะ|กิ่งไม้|กวาด|เหม็น|ปฏิกูล', c):
            sub_cat = "ขยะตกค้าง"
            dept = "ฝ่ายรักษาความสะอาดฯ"
        elif re.search(r'ทางเท้า|ฟุตบาท|บาทวิถี|ป้าย|หาบเร่|แผงลอย|กีดขวาง|จอดรถ', c):
            sub_cat = "กีดขวางทางเท้า"
            dept = "ฝ่ายเทศกิจ"
        elif re.search(r'ถนน|หลุม|บ่อ|ทรุด|ยางมะตอย|สะพาน', c):
            sub_cat = "ถนนเป็นหลุมบ่อ"
            dept = "ฝ่ายโยธา"
            
        if re.search(r'อันตรายถึงชีวิต|ตกท่อ|ไฟไหม้|แก๊สรั่ว|ทรุดตัวลึก', c): sev = 5
        elif re.search(r'อันตรายมาก|หลุมลึก|ล้ม|ขวางถนน|น้ำท่วมสูง|ด่วนมาก', c): sev = 4
        elif re.search(r'ไฟดับ|เหม็น|น้ำขัง|กระเบื้องแตก|เดือดร้อน', c): sev = 3
        
        return {"severity": sev, "sub_category": sub_cat, "predicted_dept": dept, "reason": "ประเมินโดย Heuristic Engine", "engine": "fast_heuristic"}

analyzer = LLMComplaintAnalyzer()
sample_test = "ฝาท่อระบายน้ำหน้าโรงเรียนแตกเปิดอ้า อันตรายถึงชีวิตมาก ช่วยด่วนครับ"
print("ผลการทดสอบ LLM Engine:")
print(analyzer.analyze(sample_test))


---
## 4. [TRAINING] ฝึกสอนโมเดลเดี่ยว (Single Global LightGBM Model)
ฝึกสอนโมเดล LightGBM บนข้อมูลปี 2023 (177,726 แถว) ด้วยฟังก์ชันการสูญเสีย MAE (`regression_l1`)


In [ ]:
cat_mappings = {}
train_single = train_df.copy()
val_single = val_df.copy()
test_single = test_df.copy()

for c in cat_cols:
    train_cats = sorted(list(train_single[c].dropna().unique()))
    train_single[c] = pd.Categorical(train_single[c], categories=train_cats)
    val_single[c] = pd.Categorical(val_single[c], categories=train_cats)
    test_single[c] = pd.Categorical(test_single[c], categories=train_cats)
    cat_mappings[c] = [str(x) for x in train_cats]

with open(os.path.join(MODELS_DIR, "categories.json"), "w", encoding="utf-8") as f:
    json.dump(cat_mappings, f, ensure_ascii=False, indent=2)

X_train = train_single[feature_cols]
y_train = np.log1p(train_single['duration_days'])
X_val = val_single[feature_cols]
y_val = np.log1p(val_single['duration_days'])

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=cat_cols)

params = {
    'objective': 'regression_l1', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 45, 'min_child_samples': 40,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'cat_smooth': 10.0,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1
}

print("กำลังฝึกสอน Single LightGBM Model...")
start_t = time.time()
single_model = lgb.train(params, train_data, num_boost_round=800, valid_sets=[train_data, val_data], callbacks=[lgb.early_stopping(stopping_rounds=35, verbose=False)])
print(f"--> ฝึกสอนสำเร็จในเวลา: {time.time() - start_t:.2f} วินาที (Best Iteration: {single_model.best_iteration})")

single_model.save_model(os.path.join(MODELS_DIR, "lightgbm_traffy_real.txt"))


In [ ]:
importance_gain = single_model.feature_importance(importance_type='gain')
imp_df = pd.DataFrame({'Feature': feature_cols, 'Gain': importance_gain}).sort_values(by='Gain', ascending=True)
imp_df['Gain_Pct'] = (imp_df['Gain'] / imp_df['Gain'].sum()) * 100

plt.figure(figsize=(9, 4.5))
plt.barh(imp_df['Feature'], imp_df['Gain_Pct'], color='#1f77b4')
plt.xlabel("ความสำคัญต่อการตัดสินใจ (Gain %)")
plt.title("LightGBM Feature Importance (Top Predictors of Duration)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


---
## 5. [TRAINING] ฝึกสอน Specialized Sub-models (Mixture of Experts)
ฝึกสอนโมเดลเฉพาะทางของ **5 ฝ่ายหลัก** + **1 General Fallback**


In [ ]:
TARGET_DEPTS = ['ฝ่ายโยธา', 'ฝ่ายรักษาความสะอาดฯ', 'ฝ่ายเทศกิจ', 'ฝ่ายสิ่งแวดล้อมฯ', 'สำนักการระบายน้ำ']

DEPT_SLUG_MAP = {
    'ฝ่ายโยธา': 'yotha',
    'ฝ่ายรักษาความสะอาดฯ': 'cleanliness',
    'ฝ่ายเทศกิจ': 'thetsakit',
    'ฝ่ายสิ่งแวดล้อมฯ': 'environment',
    'สำนักการระบายน้ำ': 'drainage',
}

def train_specialized_submodel(dept_name, train_sub, val_sub):
    slug = DEPT_SLUG_MAP.get(dept_name, dept_name)
    sub_cat_cols = ['district', 'main_type', 'sub_category']
    sub_feat_cols = ['district', 'main_type', 'sub_category', 'severity', 'comment_len', 'day_of_week', 'is_weekend', 'month', 'is_rainy_season', 'hour']
    
    t_df = train_sub.copy()
    v_df = val_sub.copy()
    
    sub_cats = {}
    for c in sub_cat_cols:
        cats = sorted(list(t_df[c].dropna().unique()))
        t_df[c] = pd.Categorical(t_df[c], categories=cats)
        v_df[c] = pd.Categorical(v_df[c], categories=cats)
        sub_cats[c] = [str(x) for x in cats]
        
    with open(os.path.join(SPECIALIZED_DIR, f"categories_{slug}.json"), "w", encoding="utf-8") as f:
        json.dump(sub_cats, f, ensure_ascii=False, indent=2)
        
    X_tr = t_df[sub_feat_cols]
    y_tr = np.log1p(t_df['duration_days'])
    X_va = v_df[sub_feat_cols]
    y_va = np.log1p(v_df['duration_days'])
    
    tr_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=sub_cat_cols)
    va_data = lgb.Dataset(X_va, label=y_va, reference=tr_data, categorical_feature=sub_cat_cols)
    
    n_leaves = 63 if dept_name in ['ฝ่ายโยธา', 'general'] else 31
    p = {
        'objective': 'regression_l1', 'metric': 'mae', 'boosting_type': 'gbdt',
        'learning_rate': 0.05, 'num_leaves': n_leaves, 'min_child_samples': 30,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'cat_smooth': 10.0,
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    
    m = lgb.train(p, tr_data, num_boost_round=800, valid_sets=[tr_data, va_data], callbacks=[lgb.early_stopping(stopping_rounds=35, verbose=False)])
    m.save_model(os.path.join(SPECIALIZED_DIR, f"model_{slug}.txt"))
    return m, sub_cats

print("🚀 เริ่มฝึกสอน Specialized Sub-models...")
trained_submodels = {}
for dept in TARGET_DEPTS:
    tr_s = train_df[train_df['predicted_dept'] == dept]
    va_s = val_df[val_df['predicted_dept'] == dept]
    print(f"  [+] ฝึกสอน: {dept} (Train: {len(tr_s):,} | Val: {len(va_s):,})")
    m, c = train_specialized_submodel(dept, tr_s, va_s)
    trained_submodels[dept] = m

print("  [+] ฝึกสอน: General Fallback Model...")
m_g, c_g = train_specialized_submodel("general", train_df, val_df)
trained_submodels["general"] = m_g

with open(os.path.join(SPECIALIZED_DIR, "dept_map.json"), "w", encoding="utf-8") as f:
    json.dump(DEPT_SLUG_MAP, f, ensure_ascii=False, indent=2)

print("--> [สำเร็จ] ฝึกสอนโมเดลเฉพาะทางครบทั้ง 6 โมเดลเรียบร้อยแล้ว!")


---
## 6. Specialized Router & Predictor Engine (นิยามในสมุดโค้ด)
คลาส `SpecializedBMAPredictor` ทำหน้าที่เป็น Router ส่งงานแยกตามฝ่ายรับผิดชอบ รองรับทั้งเคสเดี่ยว (<2ms) และตารางข้อมูลขนาดใหญ่แบบ Vectorized


In [ ]:
class SpecializedBMAPredictor:
    def __init__(self, models_dir: str = SPECIALIZED_DIR):
        self.models_dir = models_dir
        self.models = {}
        self.categories = {}
        self.dept_map = DEPT_SLUG_MAP.copy()
        self.feature_cols = ['district', 'main_type', 'sub_category', 'severity', 'comment_len', 'day_of_week', 'is_weekend', 'month', 'is_rainy_season', 'hour']
        self.cat_cols = ['district', 'main_type', 'sub_category']
        self._load_models()

    def _resolve_slug(self, dept_name: str) -> str:
        s = str(dept_name).strip()
        return self.dept_map.get(s, s if s in self.models else "general")

    def _load_models(self):
        dept_map_path = os.path.join(self.models_dir, "dept_map.json")
        if os.path.exists(dept_map_path):
            with open(dept_map_path, "r", encoding="utf-8") as f:
                self.dept_map.update(json.load(f))
                
        all_slugs = list(set(list(self.dept_map.values()) + ["general"]))
        for slug in all_slugs:
            m_path = os.path.join(self.models_dir, f"model_{slug}.txt")
            c_path = os.path.join(self.models_dir, f"categories_{slug}.json")
            if os.path.exists(m_path):
                self.models[slug] = lgb.Booster(model_file=m_path)
                if os.path.exists(c_path):
                    with open(c_path, "r", encoding="utf-8") as f:
                        self.categories[slug] = json.load(f)

    def is_ready(self) -> bool:
        return len(self.models) > 0

    def predict_single(self, feature_dict: dict, predicted_dept: str) -> dict:
        slug = self._resolve_slug(predicted_dept)
        model = self.models.get(slug) or self.models.get("general")
        cats = self.categories.get(slug, {})
        row_df = pd.DataFrame([feature_dict])
        
        for c in self.cat_cols:
            k_cats = cats.get(c, [])
            val = row_df[c].iloc[0]
            if val not in k_cats:
                row_df[c] = 'ทั่วไป' if 'ทั่วไป' in k_cats else (k_cats[0] if k_cats else val)
            row_df[c] = pd.Categorical(row_df[c], categories=k_cats)
            
        pred_log = model.predict(row_df[self.feature_cols])[0]
        days = float(np.expm1(pred_log))
        return {
            "predicted_days": round(max(0.1, days), 2),
            "model_used": f"Specialized ({predicted_dept})" if slug != "general" else "General Fallback",
            "routed_dept": predicted_dept,
            "model_slug": slug
        }

    def predict_df(self, df: pd.DataFrame) -> pd.DataFrame:
        out_df = df.copy()
        out_df['specialized_pred_days'] = 0.0
        out_df['model_used'] = ''
        
        dept_col = 'predicted_dept' if 'predicted_dept' in out_df.columns else None
        groups = [('general', out_df.index)] if dept_col is None else [(str(k), idxs) for k, idxs in out_df.groupby(dept_col).groups.items()]
        
        for dept_str, idxs in groups:
            slug = self._resolve_slug(dept_str)
            model = self.models.get(slug) or self.models.get("general")
            if model is None: continue
            
            cats = self.categories.get(slug, {})
            sub_df = out_df.loc[idxs, self.feature_cols].copy()
            for c in self.cat_cols:
                sub_df[c] = pd.Categorical(sub_df[c], categories=cats.get(c, []))
                
            preds_log = model.predict(sub_df[self.feature_cols])
            preds_days = np.clip(np.expm1(preds_log), 0.1, None)
            out_df.loc[idxs, 'specialized_pred_days'] = np.round(preds_days, 2)
            out_df.loc[idxs, 'model_used'] = f"Specialized ({dept_str})" if slug != "general" else "General Fallback"
            
        return out_df

predictor = SpecializedBMAPredictor()
print(f"Router พร้อมใช้งาน: {predictor.is_ready()} | รายชื่อโมเดล: {list(predictor.models.keys())}")


---
## 7. วัดผลเปรียบเทียบ Head-to-Head บน Test Set (49,010 เคส)
นำ Test Set จริงกลางปี 2024 มาเปรียบเทียบระหว่าง Single Model vs Specialized Sub-models


In [ ]:
test_eval = test_df.copy()
for c in cat_cols:
    test_eval[c] = pd.Categorical(test_eval[c], categories=cat_mappings.get(c, []))

single_preds = np.expm1(single_model.predict(test_eval[feature_cols]))
test_eval['single_pred_days'] = single_preds
test_eval = predictor.predict_df(test_eval)

actual = test_eval['duration_days'].values
s_mae = mean_absolute_error(actual, test_eval['single_pred_days'])
s_med = median_absolute_error(actual, test_eval['single_pred_days'])
p_mae = mean_absolute_error(actual, test_eval['specialized_pred_days'])
p_med = median_absolute_error(actual, test_eval['specialized_pred_days'])

s_acc3 = np.mean(np.abs(actual - test_eval['single_pred_days']) <= 3.0) * 100
p_acc3 = np.mean(np.abs(actual - test_eval['specialized_pred_days']) <= 3.0) * 100

print("=" * 65)
print("🏆 สรุปผลการแข่งขันภาพรวม (Overall Benchmark 49,010 เคส)")
print("=" * 65)
print(f"ตัวชี้วัด                      โมเดลเดี่ยว (Single)   โมเดลเฉพาะทาง (Specialized)")
print(f"1. MAE (คลาดเคลื่อนเฉลี่ย)    :    {s_mae:5.2f} วัน           {p_mae:5.2f} วัน")
print(f"2. Median AE (มัธยฐานคลาดเคลื่อน):  {s_med:5.2f} วัน           {p_med:5.2f} วัน")
print(f"3. ความแม่นยำในกรอบ ±3 วัน    :   {s_acc3:5.2f}%           {p_acc3:5.2f}%")
print("=" * 65)

dept_breakdown = []
for d in TARGET_DEPTS:
    sub = test_eval[test_eval['predicted_dept'] == d]
    if len(sub) == 0: continue
    act = sub['duration_days'].values
    sm = mean_absolute_error(act, sub['single_pred_days'])
    smed = median_absolute_error(act, sub['single_pred_days'])
    pm = mean_absolute_error(act, sub['specialized_pred_days'])
    pmed = median_absolute_error(act, sub['specialized_pred_days'])
    dept_breakdown.append({
        'หน่วยงาน': d, 'เคส': len(sub),
        'Single MAE': round(sm, 2), 'Spec MAE': round(pm, 2),
        'Single MedAE': round(smed, 2), 'Spec MedAE': round(pmed, 2),
        'MAE Diff (%)': round(((sm - pm) / sm) * 100, 2)
    })

pd.DataFrame(dept_breakdown)


---
## 8. [TIER 3] Actionable Decision Layer: จัดคิวสั่งการ กทม. (BMA Triage Matrix)
คลาส `BMADecisionLayer` ถูกนิยามไว้ใน Cell นี้โดยตรง:
$$\text{Public Impact Score} = \text{Severity (1-5)} \times \text{Density Factor (1.0-1.5)} \times \text{Predicted Days}$$
จัดกลุ่มปัญหาออกเป็น 4 Quadrants เพื่อสั่งการเจ้าหน้าที่ กทม. ทันที


In [ ]:
class BMADecisionLayer:
    def __init__(self):
        self.density_map = {
            'ป้อมปราบศัตรูพ่าย': 20751, 'สัมพันธวงศ์': 14584, 'ดินแดง': 13459, 'ธนบุรี': 11476,
            'คลองสาน': 10499, 'บางซื่อ': 9982, 'บางกอกใหญ่': 9728, 'ราชเทวี': 9173, 'บางกอกน้อย': 8672,
            'พระนคร': 8032, 'สาทร': 8017, 'บางรัก': 7978, 'ดุสิต': 7457, 'บางพลัด': 7535,
            'คลองเตย': 7500, 'ภาษีเจริญ': 6696, 'พญาไท': 6560, 'วัฒนา': 6460, 'ห้วยขวาง': 5608,
            'จตุจักร': 4594, 'ยานนาวา': 4435, 'บางกะปิ': 5100, 'บางเขน': 4200, 'ประเวศ': 3100
        }
        self.min_density = 770.0
        self.max_density = 20751.0

    def get_density_factor(self, district: str) -> float:
        clean_d = district.replace("เขต", "").strip() if isinstance(district, str) else ""
        d_val = self.density_map.get(clean_d, 5000.0)
        norm = (d_val - self.min_density) / (self.max_density - self.min_density)
        return float(np.clip(1.0 + 0.5 * norm, 1.0, 1.5))

    def triage_ticket(self, district: str, severity: int, predicted_days: float) -> dict:
        df_factor = self.get_density_factor(district)
        impact_score = severity * df_factor * max(predicted_days, 0.1)
        
        is_severe = severity >= 3
        is_fast = predicted_days <= 5.0
        
        if is_severe and is_fast:
            quadrant = "Critical Urgent"
            action = "สั่งหน่วยเคลื่อนที่เร็วเข้าเคลียร์หน้างานภายใน 24 ชม. ทันที"
        elif is_severe and not is_fast:
            quadrant = "High Impact Project"
            action = "ตั้งงบด่วนหรือระดมช่างข้ามเขตเพื่อลดระยะเวลาซ่อม"
        elif not is_severe and is_fast:
            quadrant = "Quick Win"
            action = "ส่งทีมเขตเข้าเก็บงานทันทีเพื่อเพิ่มคะแนนความพึงพอใจ"
        else:
            quadrant = "Routine Maintenance"
            action = "ส่งเข้าคิวงานซ่อมบำรุงประจำวันตามรอบปกติ"
            
        return {"impact_score": round(impact_score, 2), "quadrant": quadrant, "action": action}

    def evaluate_df(self, df: pd.DataFrame, duration_col='specialized_pred_days') -> pd.DataFrame:
        res_df = df.copy()
        scores, quads, acts = [], [], []
        for _, row in res_df.iterrows():
            t = self.triage_ticket(str(row.get('district', '')), int(row.get('severity', 3)), float(row[duration_col]))
            scores.append(t['impact_score'])
            quads.append(t['quadrant'])
            acts.append(t['action'])
        res_df['public_impact_score'] = scores
        res_df['triage_quadrant'] = quads
        res_df['bma_action'] = acts
        return res_df

decision_layer = BMADecisionLayer()
sample_triage = decision_layer.evaluate_df(test_eval.head(1000))
print("สัดส่วนปัญหาในแต่ละ Quadrant (1,000 เคสตัวอย่าง):")
print(sample_triage['triage_quadrant'].value_counts())


---
## 9. 🚀 End-to-End Simulation (ข้อความภาษาไทยสดๆ -> LLM -> AI Model -> Action)
ทดสอบระบบครบ 3 ชั้นแบบสดๆ:
1. **Tier 1 (LLM):** อ่านข้อความดิบ -> สกัด Severity, Sub-category, และ Department
2. **Tier 2 (MoE Model):** Router เลือกโมเดลเฉพาะทาง -> ทำนายวันแล้วเสร็จ
3. **Tier 3 (Decision Layer):** คำนวณ Impact Score -> สั่งการ กทม. ทันที!


In [ ]:
def submit_complaint_end_to_end(district: str, main_type: str, raw_comment: str, hour: int = 9):
    print("=" * 70)
    print("📥 [INPUT] ได้รับเรื่องร้องเรียนใหม่จากประชาชน")
    print("=" * 70)
    print(f"พื้นที่แจ้งเหตุ : เขต{district}")
    print(f"หมวดปัญหาหลัก  : {main_type}")
    print(f"ข้อความประชาชน : '{raw_comment}'")
    
    # 1. LLM Feature Extraction
    print("
[Step 1] ให้ Local LLM (Ollama / Heuristic) วิเคราะห์ข้อความ...")
    llm_res = analyzer.analyze(raw_comment)
    sev = llm_res['severity']
    sub_cat = llm_res['sub_category']
    dept = llm_res['predicted_dept']
    
    print(f"  -> สกัดความรุนแรง (Severity) : ระดับ {sev}/5")
    print(f"  -> ระบุเนื้องานย่อย (Sub-type) : {sub_cat}")
    print(f"  -> ส่งต่อฝ่ายรับผิดชอบ         : {dept}")
    print(f"  -> เหตุผลประกอบ              : {llm_res['reason']}")
    print(f"  -> Engine ที่ประมวลผล         : {llm_res['engine']}")
    
    # 2. Specialized ML Prediction
    print("
[Step 2] Route เรื่องเข้าสู่ Specialized Sub-model ประจำหน่วยงาน...")
    payload = {
        'district': district, 'main_type': main_type, 'sub_category': sub_cat,
        'severity': sev, 'comment_len': len(raw_comment), 'day_of_week': 2,
        'is_weekend': 0, 'month': 6, 'is_rainy_season': 1, 'hour': hour
    }
    pred_res = predictor.predict_single(payload, predicted_dept=dept)
    pred_days = pred_res['predicted_days']
    print(f"  -> โมเดลที่ถูกเลือกใช้ : {pred_res['model_used']} (Slug: {pred_res['model_slug']})")
    print(f"  -> คาดการณ์ระยะเวลาซ่อม : {pred_days} วัน (~{pred_days*24:.1f} ชั่วโมง)")
    
    # 3. Actionable Triage Matrix
    print("
[Step 3] คำนวณ Public Impact Score และจัดกลุ่ม Triage สั่งการ...")
    triage = decision_layer.triage_ticket(district, sev, pred_days)
    print(f"  -> Public Impact Score : {triage['impact_score']}")
    print(f"  -> กลุ่ม Triage กทม.    : 🚨 {triage['quadrant']}")
    print(f"  -> คำสั่งการผู้บริหาร กทม.: 📢 {triage['action']}")
    print("=" * 70)

# ทดสอบรันเคสตัวอย่าง 1: ปัญหาวิกฤตอันตรายถึงชีวิต
submit_complaint_end_to_end(
    district="จตุจักร", main_type="ถนน",
    raw_comment="ฝาท่อระบายน้ำหน้าตลาดแตกเปิดอ้ากว้างมาก มอเตอร์ไซค์สะดุดล้มไปแล้วเมื่อเช้า อันตรายถึงชีวิต ช่วยซ่อมด่วนครับ",
    hour=8
)

# ทดสอบรันเคสตัวอย่าง 2: ปัญหาขยะตกค้าง
submit_complaint_end_to_end(
    district="สาทร", main_type="ขยะ",
    raw_comment="มีคนนำขยะชิ้นใหญ่และกิ่งไม้มาทิ้งกองไว้ริมทางเท้า ส่งกลิ่นเหม็นและกีดขวางคนเดิน",
    hour=14
)
